# 🏭 Project 14.4 — Integrated Control System
**DSA for Mechatronics · Week 14 — Review & Final Project**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.

**Topics integrated:** LRU Cache / Hash Map + Linked List (W05/W06) · Dijkstra Shortest Path (W13) · Kruskal's MST + Union-Find (W13) · Merge K Sorted Streams (W08 Heap) · Top-K query (W08)


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq, math
from collections import defaultdict, deque, Counter, OrderedDict
from functools import lru_cache
import bisect

_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A factory control system must handle four data-management challenges:

1. **LRU Cache** — a sensor reading cache with limited capacity; when full,
   evict the **least recently used** entry (hash map + doubly linked list → O(1) get & put)
2. **Dijkstra shortest path** — route a maintenance robot through the factory
   graph to its target using minimum total travel cost
3. **Kruskal's MST with union-find** — determine the minimum-cost wiring
   to connect all sensor nodes, using union-find for O(α) cycle detection
4. **Merge k sorted streams** — fuse readings from k sorted sensor streams
   into one sorted stream (min-heap of size k → O(n log k))


## Step 1 — Generate all datasets

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
LRU_CAP    = random.randint(3, 5)
LRU_OPS    = random.randint(16, 24)
N_GRAPH    = random.randint(7, 11)
EDGE_P     = random.uniform(0.40, 0.60)
W_MAX      = random.randint(15, 25)
N_STREAMS  = random.randint(3, 5)
STREAM_LEN = random.randint(5, 8)

# LRU operations: mix of GET and PUT
KEY_POOL = list(range(1, 10))
LRU_SEQ  = []
for _ in range(LRU_OPS):
    op = random.choice(["GET", "PUT"])
    k  = random.choice(KEY_POOL)
    v  = random.randint(10, 99) if op == "PUT" else None
    LRU_SEQ.append((op, k, v))

# Factory graph (undirected weighted, guaranteed connected)
G_EDGES  = []
shuffled = list(range(N_GRAPH)); random.shuffle(shuffled)
for i in range(1, N_GRAPH):
    G_EDGES.append((shuffled[i-1], shuffled[i], random.randint(1, W_MAX)))
for u in range(N_GRAPH):
    for v in range(u+1, N_GRAPH):
        if random.random() < EDGE_P:
            if not any((a==u and b==v) or (a==v and b==u) for a,b,_ in G_EDGES):
                G_EDGES.append((u, v, random.randint(1, W_MAX)))
G_ADJ = defaultdict(list)
for u, v, w in G_EDGES:
    G_ADJ[u].append((w, v)); G_ADJ[v].append((w, u))

G_SRC, G_DST = 0, N_GRAPH - 1

# k sorted streams
STREAMS = [sorted(random.randint(1, 80) for _ in range(STREAM_LEN))
           for _ in range(N_STREAMS)]

print(f"Control system parameters:")
print(f"  LRU cache capacity : {LRU_CAP}  ({LRU_OPS} operations)")
print(f"  Factory graph      : {N_GRAPH} nodes, {len(G_EDGES)} edges  ({G_SRC}→{G_DST})")
print(f"  Graph edges        : {sorted(G_EDGES)}")
print()
print(f"  Sensor streams ({N_STREAMS} streams × {STREAM_LEN} readings):")
for i, s in enumerate(STREAMS):
    print(f"    Stream {i}: {s}")


## Step 2 — LRU Cache (hash map + doubly linked list)

In [ ]:
class DNode:
    """Doubly-linked list node."""
    def __init__(self, key=0, val=0):
        self.key  = key
        self.val  = val
        self.prev = None
        self.next = None

class LRUCache:
    """
    LRU Cache with O(1) get and put.

    Data structure: hash map + doubly linked list.
      - Hash map: key → DNode  (O(1) lookup)
      - Doubly linked list: most-recently used at head, LRU at tail.
        Dummy head and tail nodes simplify edge cases.

    get(key):
      If key exists → move its node to head (mark as recently used) → return val.
      Otherwise → return -1.

    put(key, val):
      If key exists → update val and move to head.
      If new key:
        - Add new node at head.
        - If over capacity → remove tail node (LRU eviction) + delete from map.

    Moving to head and removing tail are both O(1) because we have prev/next pointers.
    """
    def __init__(self, capacity):
        self.cap  = capacity
        self.map  = {}
        self.head = DNode()   # dummy head (most recent)
        self.tail = DNode()   # dummy tail (least recent / LRU)
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_head(self, node):
        node.next = self.head.next
        node.prev = self.head
        self.head.next.prev = node
        self.head.next      = node

    def get(self, key):
        if key not in self.map: return -1
        node = self.map[key]
        self._remove(node)
        self._add_to_head(node)
        return node.val

    def put(self, key, val):
        if key in self.map:
            node = self.map[key]
            node.val = val
            self._remove(node)
            self._add_to_head(node)
        else:
            node = DNode(key, val)
            self.map[key] = node
            self._add_to_head(node)
            if len(self.map) > self.cap:
                lru = self.tail.prev
                self._remove(lru)
                del self.map[lru.key]
                return lru.key   # evicted key
        return None

    def state(self):
        """Return cache contents MRU→LRU order."""
        res = []; node = self.head.next
        while node != self.tail:
            res.append((node.key, node.val)); node = node.next
        return res

cache    = LRUCache(LRU_CAP)
lru_log  = []
evictions = 0

for op, k, v in LRU_SEQ:
    if op == "PUT":
        evicted = cache.put(k, v)
        if evicted is not None: evictions += 1
        lru_log.append(f"PUT({k},{v:2d}) {'→ evict key '+str(evicted) if evicted else '           '}  cache={cache.state()}")
    else:
        result = cache.get(k)
        lru_log.append(f"GET({k})     → {result if result != -1 else 'MISS':3}             cache={cache.state()}")

hits   = sum(1 for op,k,v in LRU_SEQ if op=="GET" and cache.get(k) != -1)
misses = sum(1 for op,k,v in LRU_SEQ if op=="GET" and cache.get(k) == -1)
total_gets = sum(1 for op,k,v in LRU_SEQ if op=="GET")

print(f"LRU Cache simulation (capacity={LRU_CAP}):")
print()
for line in lru_log[:min(18, LRU_OPS)]:
    print(f"  {line}")
if LRU_OPS > 18:
    print(f"  ... ({LRU_OPS-18} more operations)")
print()
final_state = cache.state()
print(f"  Final cache state : {final_state}")
print(f"  Total ops         : {LRU_OPS}  (GET={total_gets}, PUT={LRU_OPS-total_gets})")
print(f"  Evictions         : {evictions}")


## Step 3 — Dijkstra shortest path + Kruskal's MST

In [ ]:
def dijkstra(n, adj, src, dst):
    dist = {v: float('inf') for v in range(n)}
    prev = {v: None for v in range(n)}
    dist[src] = 0
    heap = [(0, src)]
    vis  = set()
    while heap:
        d, u = heapq.heappop(heap)
        if u in vis: continue
        vis.add(u)
        if u == dst: break
        for w, v in adj[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(heap, (dist[v], v))
    if dist[dst] == float('inf'): return -1, []
    path = []; node = dst
    while node is not None:
        path.append(node); node = prev[node]
    return dist[dst], list(reversed(path))

class UF:
    def __init__(self, n):
        self.p = list(range(n)); self.r = [0]*n
    def find(self, x):
        if self.p[x] != x: self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry: return False
        if self.r[rx] < self.r[ry]: rx, ry = ry, rx
        self.p[ry] = rx
        if self.r[rx] == self.r[ry]: self.r[rx] += 1
        return True

def kruskal(n, edges):
    uf = UF(n); mst = []; tw = 0
    for w, u, v in sorted((w,u,v) for u,v,w in edges):
        if uf.union(u, v):
            mst.append((w, u, v)); tw += w
            if len(mst) == n-1: break
    return mst, tw

dijk_dist, dijk_path = dijkstra(N_GRAPH, G_ADJ, G_SRC, G_DST)
mst_edges, mst_weight = kruskal(N_GRAPH, G_EDGES)

# Verify Dijkstra path
if dijk_dist >= 0:
    path_wt = sum(next(w for a,b,w in G_EDGES
                        if (a==dijk_path[i] and b==dijk_path[i+1]) or
                           (b==dijk_path[i] and a==dijk_path[i+1]))
                  for i in range(len(dijk_path)-1))
    dijk_ok = path_wt == dijk_dist
else:
    path_wt = 0; dijk_ok = True

# Verify MST connectivity
from collections import deque as _dq
def bfs_conn(n, edges):
    adj2 = defaultdict(list)
    for w,u,v in edges: adj2[u].append(v); adj2[v].append(u)
    vis = set(); q = _dq([0])
    while q:
        node = q.popleft()
        if node in vis: continue
        vis.add(node); q.extend(adj2[node])
    return len(vis) == n

mst_connected = bfs_conn(N_GRAPH, mst_edges)

print(f"Dijkstra shortest path ({G_SRC}→{G_DST}):")
if dijk_dist >= 0:
    print(f"  Path     : {' → '.join(map(str, dijk_path))}")
    print(f"  Distance : {dijk_dist}  (sum={path_wt})  {'✅' if dijk_ok else '❌'}")
else:
    print(f"  No path found")

print()
print(f"Kruskal's MST ({N_GRAPH} nodes, {len(G_EDGES)} edges):")
for w,u,v in mst_edges:
    print(f"  w={w:3d}  {u}─{v}")
print(f"  Total MST weight  : {mst_weight}")
print(f"  Edges (= n-1={N_GRAPH-1}) : {len(mst_edges)}  {'✅' if len(mst_edges)==N_GRAPH-1 else '❌'}")
print(f"  MST connected     : {'✅ YES' if mst_connected else '❌ NO'}")
print(f"  Total graph wt    : {sum(w for _,_,w in G_EDGES)}  → saved {sum(w for _,_,w in G_EDGES)-mst_weight}")


## Step 4 — Merge k sorted sensor streams (min-heap)

In [ ]:
def merge_k_sorted(streams):
    """
    Merge k sorted arrays into one sorted array.

    Min-heap approach:
      1. Push (value, stream_index, element_index) for the FIRST element
         of each stream into a min-heap.
      2. Pop the minimum element → add to result.
      3. If that stream has more elements, push the next one.
      4. Repeat until heap is empty.

    Correctness: the heap always holds the current minimum candidate
    from each stream. Popping gives the global minimum among all
    "current fronts".

    O(n log k) total — n = total elements, k = number of streams.
    Much better than concatenate-and-sort O(n log n) when k << n.
    """
    heap   = []
    result = []
    for si, stream in enumerate(streams):
        if stream:
            heapq.heappush(heap, (stream[0], si, 0))
    while heap:
        val, si, ei = heapq.heappop(heap)
        result.append(val)
        if ei + 1 < len(streams[si]):
            heapq.heappush(heap, (streams[si][ei+1], si, ei+1))
    return result

merged = merge_k_sorted(STREAMS)
all_flat = sorted(v for s in STREAMS for v in s)
merge_ok = merged == all_flat

n_total = sum(len(s) for s in STREAMS)

print(f"Merge {N_STREAMS} sorted streams (total {n_total} elements):")
print()
for i, s in enumerate(STREAMS):
    print(f"  Stream {i}: {s}")
print()
print(f"  Merged result  : {merged}")
print(f"  Sort-verify    : {all_flat}")
print(f"  Correct        : {'✅ YES' if merge_ok else '❌ NO'}")
print()
print(f"  Complexity: O(n log k) = O({n_total} × log {N_STREAMS}) ≈ "
      f"O({int(n_total * math.log2(max(N_STREAMS, 2)))}) operations")
print(f"  vs concatenate+sort: O(n log n) ≈ "
      f"O({int(n_total * math.log2(max(n_total, 2)))})")
if N_STREAMS > 1:
    print(f"  Speedup ratio ≈ {math.log2(max(n_total,2)) / math.log2(max(N_STREAMS,2)):.2f}×")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " INTEGRATED CONTROL SYSTEM — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  LRU CACHE" + " "*(W-11) + "║")
print(f"║  {'Capacity':<28}: {LRU_CAP:<26}║")
print(f"║  {'Total operations':<28}: {LRU_OPS:<26}║")
print(f"║  {'Evictions':<28}: {evictions:<26}║")
print(f"║  {'Final state (MRU→LRU)':<28}: {str(final_state):<26}║")
print("╠" + "═"*W + "╣")
print("║  DIJKSTRA SHORTEST PATH" + " "*(W-24) + "║")
print(f"║  {'Graph nodes':<28}: {N_GRAPH:<26}║")
print(f"║  {'Graph edges':<28}: {len(G_EDGES):<26}║")
if dijk_dist >= 0:
    print(f"║  {'Shortest distance':<28}: {dijk_dist:<26}║")
    print(f"║  {'Path':<28}: {' → '.join(map(str,dijk_path)):<26}║")
    print(f"║  {'Path weight verified':<28}: {'YES ✅' if dijk_ok else 'NO ❌':<26}║")
else:
    print(f"║  {'Result':<28}: {'No path found':<26}║")
print("╠" + "═"*W + "╣")
print("║  KRUSKAL'S MST (UNION-FIND)" + " "*(W-28) + "║")
print(f"║  {'MST edges':<28}: {len(mst_edges)} (= n-1 = {N_GRAPH-1}){'':<10}║")
print(f"║  {'MST total weight':<28}: {mst_weight:<26}║")
print(f"║  {'Cable saved':<28}: {sum(w for _,_,w in G_EDGES)-mst_weight:<26}║")
print(f"║  {'Connected':<28}: {'YES ✅' if mst_connected else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  MERGE K SORTED STREAMS" + " "*(W-24) + "║")
print(f"║  {'Streams (k)':<28}: {N_STREAMS:<26}║")
print(f"║  {'Total elements (n)':<28}: {n_total:<26}║")
print(f"║  {'Merge correct':<28}: {'YES ✅' if merge_ok else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")
print()
print("  🎓  Congratulations on completing the DSA course!")
print("  All topics from W01–W13 now live in your toolbox.")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What do they tell you about the algorithms in this project?

*Your answer here:*

---

**Q2 — Which technique was most surprising, and why?**
Pick one algorithm from this project. Explain why the approach works — and give one scenario where it would *fail* or need modification.

*Your answer here:*

---

**Q3 — How does this project connect to earlier weeks?**
Name at least two specific weeks whose topics appear in this project and explain the connection.

*Your answer here:*
